<a href="https://colab.research.google.com/github/tettyvera/loan-portfolio-risk-analytics/blob/main/Loan_Portfolio_SQL_Exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loan Portfolio SQL Exploration

**Portfolio:** Loan Portfolio Performance & Risk Analytics  
**Tools:** Google Colab + DuckDB SQL  
**Analysis snapshot:** 2026-08-31  
**Currency:** IDR

This notebook uses **fictional/synthetic** CSV files, not confidential employer data.

## 1 — Install and import the SQL engine

In [3]:
%pip -q install duckdb
import duckdb
print("DuckDB is ready:", duckdb.__version__)

DuckDB is ready: 1.3.2


## 2 — Upload the five CSVs


In [4]:
from google.colab import files
uploaded = files.upload()
required = {"borrowers.csv", "lenders.csv", "loans.csv", "installments.csv", "repayments.csv"}
missing = required - set(uploaded)
if missing:
    raise ValueError(f"Missing files: {sorted(missing)}. Run this cell again and upload all five CSVs.")
print("All five dataset files uploaded successfully.")

Saving borrowers.csv to borrowers.csv
Saving installments.csv to installments.csv
Saving lenders.csv to lenders.csv
Saving loans.csv to loans.csv
Saving repayments.csv to repayments.csv
All five dataset files uploaded successfully.


## 3 — Create SQL views


In [5]:
con = duckdb.connect(database=":memory:")
for name in ["borrowers", "lenders", "loans", "installments", "repayments"]:
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv_auto('{name}.csv')")
print("SQL views ready: borrowers, lenders, loans, installments, repayments")

SQL views ready: borrowers, lenders, loans, installments, repayments


## 4 — Preview the raw loan data


In [6]:
con.sql("""
SELECT *
FROM loans
LIMIT 5;
""").df()

,loan_id,borrower_id,lender_id,disbursement_date,principal_amount,tenor_months
0,LN000001,B01102,L002,2026-07-07,28300000.0,12
1,LN000002,B00122,L003,2024-05-28,16900000.0,6
2,LN000003,B00086,L003,2025-09-28,11200000.0,12
3,LN000004,B01340,L001,2025-10-22,21600000.0,18
4,LN000005,B00721,L003,2024-11-08,5400000.0,18


## 5 — Check how many rows each table contains

In [7]:
con.sql("""
SELECT 'borrowers' AS table_name, COUNT(*) AS total_rows FROM borrowers
UNION ALL
SELECT 'lenders', COUNT(*) FROM lenders
UNION ALL
SELECT 'loans', COUNT(*) FROM loans
UNION ALL
SELECT 'installments', COUNT(*) FROM installments
UNION ALL
SELECT 'repayments', COUNT(*) FROM repayments;
""").df()

,table_name,total_rows
0,borrowers,1500
1,lenders,4
2,loans,2000
3,installments,19458
4,repayments,13641


## 6 — Business question: Which lending partner disbursed the most principal?

In [8]:
con.sql("""
SELECT
    ld.lender_name,
    COUNT(*) AS total_loans,
    SUM(l.principal_amount) AS total_disbursement_idr,
    ROUND(AVG(l.principal_amount), 0) AS avg_loan_size_idr
FROM loans AS l
JOIN lenders AS ld
    ON l.lender_id = ld.lender_id
WHERE CAST(l.disbursement_date AS DATE) <= DATE '2026-08-31'
GROUP BY ld.lender_name
ORDER BY total_disbursement_idr DESC;
""").df()

,lender_name,total_loans,total_disbursement_idr,avg_loan_size_idr
0,Meridian Capital,789,1.250000e+10,15842839.0
1,Crescent Lending,533,8.381800e+09,15725704.0
2,Harbor Finance,417,6.575000e+09,15767386.0
3,Summit Credit,261,4.247600e+09,16274330.0


## 7 — Business question: How did monthly disbursement change?

In [9]:
con.sql("""
SELECT
    DATE_TRUNC('month', CAST(disbursement_date AS DATE)) AS disbursement_month,
    COUNT(*) AS loan_count,
    SUM(principal_amount) AS disbursement_idr
FROM loans
WHERE CAST(disbursement_date AS DATE) <= DATE '2026-08-31'
GROUP BY 1
ORDER BY 1;
""").df()

,disbursement_month,loan_count,disbursement_idr
0,2024-01-01,53,8.976000e+08
1,2024-02-01,68,1.115300e+09
2,2024-03-01,64,1.004800e+09
3,2024-04-01,57,8.737000e+08
4,2024-05-01,65,1.068500e+09
5,2024-06-01,55,9.001000e+08
6,2024-07-01,61,9.381000e+08
7,2024-08-01,75,1.157100e+09
8,2024-09-01,59,9.709000e+08
9,2024-10-01,58,9.920000e+08


## 8 — Data quality: find missing borrower or lender references

In [10]:
con.sql("""
SELECT
  SUM(CASE WHEN b.borrower_id IS NULL THEN 1 ELSE 0 END) AS loans_without_borrower,
  SUM(CASE WHEN ld.lender_id IS NULL THEN 1 ELSE 0 END) AS loans_without_lender
FROM loans AS l
LEFT JOIN borrowers AS b ON l.borrower_id = b.borrower_id
LEFT JOIN lenders AS ld ON l.lender_id = ld.lender_id;
""").df()

,loans_without_borrower,loans_without_lender
0,0.0,0.0


## 9 — Important: Avoid a misleading JOIN

Joining `loans` directly to `repayments` and then summing `loans.principal_amount` **overcounts** disbursements when one loan has multiple repayments. In later lessons we'll aggregate repayments at loan level *before* joining to loans.